# Simulador de reclamaciones nuevas

Genera un lote controlado de reclamaciones para probar el pipeline incremental.

Este notebook representa temporalmente al sistema fuente. No pertenece al proceso productivo de scoring.

In [0]:
from pyspark.sql import functions as F

INPUT_TABLE = (
    "workspace.fraude_prod."
    "reclamaciones_entrada_demo"
)

BATCH_ID = "DEMO-INC-001"
BATCH_PREFIX = f"{BATCH_ID}-"

BATCH_SIZE = 100

print("Tabla destino:", INPUT_TABLE)
print("Lote:", BATCH_ID)
print("Tamaño esperado:", BATCH_SIZE)

Tabla destino: workspace.fraude_prod.reclamaciones_entrada_demo
Lote: DEMO-INC-001
Tamaño esperado: 100


In [0]:
existing_df = spark.table(INPUT_TABLE)

# Excluir registros creados previamente
# por este mismo simulador.
base_candidates_df = (
    existing_df
    .filter(
        ~F.col("id_reclamacion")
        .startswith("DEMO-INC-")
    )
)

candidate_df = (
    base_candidates_df
    .orderBy("id_reclamacion")
    .limit(BATCH_SIZE)
)

new_batch_df = (
    candidate_df
    .withColumn(
        "id_reclamacion",
        F.concat(
            F.lit(BATCH_PREFIX),
            F.substring(
                F.sha2(
                    F.col("id_reclamacion"),
                    256,
                ),
                1,
                24,
            ),
        ),
    )
    .withColumn(
        "fecha_ingesta",
        F.current_timestamp(),
    )
)

display(
    new_batch_df.select(
        "id_reclamacion",
        "fecha_ingesta",
    ).limit(10)
)

id_reclamacion,fecha_ingesta
DEMO-INC-001-4655386a9e379c13fbef2466,2026-09-19T04:00:26.562Z
DEMO-INC-001-c5243d3c1c77e9f34808c51d,2026-09-19T04:00:26.562Z
DEMO-INC-001-db177e55456d6feab8895a70,2026-09-19T04:00:26.562Z
DEMO-INC-001-2b498ca24f8fae563fb8109a,2026-09-19T04:00:26.562Z
DEMO-INC-001-b493beaae53c056caac2aa3f,2026-09-19T04:00:26.562Z
DEMO-INC-001-6f0737454ca1fbac52def03f,2026-09-19T04:00:26.562Z
DEMO-INC-001-3cd390a9b5448ca0b948eaf1,2026-09-19T04:00:26.562Z
DEMO-INC-001-fb06b9096c4bc65b9514d1f9,2026-09-19T04:00:26.562Z
DEMO-INC-001-628fbaeadcd296b06a7574d0,2026-09-19T04:00:26.562Z
DEMO-INC-001-b735d5dbdcdb24b6b931dfd1,2026-09-19T04:00:26.562Z


In [0]:
existing_ids_df = (
    existing_df
    .select("id_reclamacion")
)

new_batch_to_insert_df = (
    new_batch_df
    .join(
        existing_ids_df,
        on="id_reclamacion",
        how="left_anti",
    )
)

NEW_RECORDS = (
    new_batch_to_insert_df.count()
)

if NEW_RECORDS > 0:
    (
        new_batch_to_insert_df
        .select(*existing_df.columns)
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(INPUT_TABLE)
    )

print(
    "Reclamaciones nuevas insertadas:",
    NEW_RECORDS,
)

TOTAL_INPUT_RECORDS = (
    spark.table(INPUT_TABLE).count()
)

print(
    "Total en la tabla de entrada:",
    TOTAL_INPUT_RECORDS,
)

Reclamaciones nuevas insertadas: 100
Total en la tabla de entrada: 12876


In [0]:
inserted_batch_df = (
    spark.table(INPUT_TABLE)
    .filter(
        F.col("id_reclamacion")
        .startswith(BATCH_PREFIX)
    )
)

INSERTED_BATCH_COUNT = (
    inserted_batch_df.count()
)

if INSERTED_BATCH_COUNT != BATCH_SIZE:
    raise ValueError(
        "El lote no contiene la cantidad "
        "esperada. "
        f"Esperado={BATCH_SIZE}, "
        f"Encontrado={INSERTED_BATCH_COUNT}"
    )

duplicate_ids = (
    inserted_batch_df
    .groupBy("id_reclamacion")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_ids > 0:
    raise ValueError(
        "El lote contiene identificadores "
        "duplicados"
    )

print("Lote verificado correctamente")
print("Registros:", INSERTED_BATCH_COUNT)
print("Duplicados: 0")

Lote verificado correctamente
Registros: 100
Duplicados: 0
